In [1]:
# ============================================================
# WEEK 8 BBO OPTIMISATION NOTEBOOK
# Advanced Bayesian + Neural Optimisation Strategy
# ============================================================

import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from sklearn.preprocessing import StandardScaler

# ============================================================
# WEEK 1–7 INPUT DATA
# ============================================================

week_inputs = {

1: [
[0.211325,0.788675],
[0.654299,0.353479],
[0.582812,0.421077],
[0.183704,0.127166],
[0.582812,0.421077],
[0.582812,0.421077],
[0.582812,0.421077]
],

2: [
[0.723607,0.276393],
[0.754299,0.253479],
[0.712865,0.284413],
[0.255353,0.749841],
[0.684812,0.281383],
[0.996744,0.999561],
[0.744767,0.322712]
],

3: [
[0.166667,0.5,0.833333],
[0.304299,0.553479,0.709691],
[0.118496,0.481282,0.876608],
[0.094906,0.770362,0.859589],
[0.171685,0.508728,0.819757],
[0.166667,0.5,0.833333],
[0.167431,0.503246,0.833129]
],

4: [
[0.125,0.375,0.625,0.875],
[0.804299,0.603479,0.409691,0.205016],
[1.0,0.683447,0.334333,0.0],
[0.820856,0.226975,0.784625,0.113609],
[0.734042,0.687625,0.466427,0.303264],
[0.743137,0.718436,0.581810,0.365984],
[0.692844,0.522126,0.400511,0.337375]
],

5: [
[0.875,0.625,0.375,0.125],
[0.854299,0.653479,0.359691,0.155016],
[0.882245,0.615032,0.380358,0.114494],
[0.917860,0.273128,0.475575,0.052446],
[0.568707,0.909615,0.957143,0.011420],
[0.572707,0.914615,0.952143,0.016420],
[0.543363,0.994173,0.995000,0.003824]
],

6: [
[0.15,0.35,0.55,0.75,0.95],
[0.904299,0.703479,0.509691,0.305016,0.103275],
[0.0,0.226282,0.564108,0.905744,1.0],
[0.052509,0.789636,0.354234,0.228337,0.889991],
[0.744968,0.084644,0.801527,0.169728,0.990511],
[0.236209,0.379873,0.584786,0.843391,1.0],
[0.236209,0.379873,0.584786,0.843391,1.0]
],

7: [
[0.9,0.1,0.7,0.3,0.5,0.8],
[0.884299,0.123479,0.729691,0.285016,0.523275,0.787492],
[0.905495,0.091782,0.689608,0.305244,0.491854,0.804378],
[0.833796,0.096195,0.232612,0.789158,0.185769,0.681185],
[1.0,0.229371,0.690054,0.244152,0.521424,1.0],
[0.024461,0.059677,0.995330,0.299568,0.038576,0.691027],
[0.194327,0.008117,0.940397,0.331469,0.120428,0.708672]
],

8: [
[0.111111,0.222222,0.333333,0.444444,0.555555,0.666667,0.777778,0.888889],
[0.104299,0.243479,0.319691,0.465016,0.573275,0.647492,0.794889,0.860861],
[0.113495,0.214782,0.338108,0.437244,0.549353,0.673378,0.771789,0.898699],
[0.448564,0.335191,0.751694,0.277261,0.167614,0.717472,0.304117,0.767059],
[0.022620,0.0,0.288709,0.762817,0.551656,0.961269,0.843087,1.0],
[0.148189,0.189483,0.270757,0.392507,0.446351,0.670593,0.665417,1.0],
[0.208946,0.148993,0.202639,0.316908,0.431312,0.503474,0.634079,0.995000]
]
}

# ============================================================
# WEEK 1–7 OUTPUTS
# ============================================================

week_outputs = {
1:[1.1327056288856873e-125,
1.1867858443665185e-41,
7.99676981308551e-19,
-2.410121917336285e-100,
7.99676981308551e-19,
7.99676981308551e-19,
7.99676981308551e-19],

2:[0.5675786892822564,
0.2715258567299176,
0.5213723465552891,
0.0244939290195959,
0.4258127251524913,
-0.004122974640885927,
0.3799297936947829],

3:[-0.032385408076210126,
-0.1198581070659559,
-0.04726977098841539,
-0.04207093453964322,
-0.06232295859499999,
-0.032385408076210126,
-0.03240117791304375],

4:[-17.20744048260943,
-13.082213230390916,
-25.67625344929702,
-20.330739763644917,
-12.496845976106417,
-14.192389833871584,
-6.92709780967855],

5:[60.223125,
50.179390256321376,
64.78245026474816,
61.278397876784794,
942.2521944777988,
943.6841142765069,
1723.425126888365],

6:[-1.3287857969718009,
-1.5080002951000964,
-1.7372122723701597,
-2.4624737429462145,
-2.280900502246122,
-1.2257941580841207,
-1.2923495282910902],

7:[0.34356041660314957,
0.31639485635485903,
0.3507828450585503,
0.0634803557047261,
0.21828066675598462,
0.4410410448155631,
0.919847131115406],

8:[9.0501517903694,
9.0219949134074,
9.0587238074074,
8.275283250689899,
8.427686115001,
9.3346877420455,
9.472143824862]
}

# ============================================================
# GENERATE WEEK 8 PREDICTIONS
# ============================================================

np.random.seed(42)

week8_queries = {}

for fn in range(1,9):

    X = np.array(week_inputs[fn])
    y = np.array(week_outputs[fn])

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Gaussian Process surrogate
    kernel = Matern(nu=2.5)

    gp = GaussianProcessRegressor(
        kernel=kernel,
        alpha=1e-6,
        normalize_y=True,
        random_state=42
    )

    gp.fit(X_scaled, y)

    # Find best historical point
    best_idx = np.argmax(y)
    best_point = X[best_idx]

    # Exploitation stronger for Function 5
    if fn == 5:
        noise = np.random.normal(0,0.015,size=len(best_point))
    else:
        noise = np.random.normal(0,0.05,size=len(best_point))

    candidate = best_point + noise

    # Clip between 0 and 1
    candidate = np.clip(candidate,0,1)

    # Round to 6 decimals
    candidate = np.round(candidate,6)

    week8_queries[fn] = candidate

# ============================================================
# PRINT WEEK 8 SUBMISSIONS
# ============================================================

print("================================================")
print("WEEK 8 BBO SUBMISSION VALUES")
print("================================================\n")

for fn,val in week8_queries.items():

    formatted = "-".join([f"{x:.6f}" for x in val])

    print(f"Function {fn}:")
    print(formatted)
    print()

WEEK 8 BBO SUBMISSION VALUES

Function 1:
0.607648-0.414164

Function 2:
0.755991-0.352544

Function 3:
0.154959-0.488293-0.912294

Function 4:
0.731216-0.498652-0.427639-0.314204

Function 5:
0.536377-0.997802-0.966301-0.000000

Function 6:
0.208095-0.329231-0.600498-0.797990-0.929385

Function 7:
0.267609-0.000000-0.943773-0.260232-0.093209-0.714218

Function 8:
0.151396-0.167778-0.172607-0.302323-0.401227-0.596088-0.633404-0.942114



/opt/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


In [2]:
# ============================================================
# WEEK 8 BBO STRATEGY
# Incorporating Week 1–7 input and output files
# Bayesian Optimisation + Adaptive Exploitation/Exploration
# ============================================================

import ast
import re
import numpy as np
import warnings

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RBF, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(42)

# ============================================================
# 1. LOAD INPUT AND OUTPUT FILES
# ============================================================

with open("inputs-3.txt", "r", encoding="utf-8") as f:
    inputs_text = f.read()

with open("outputs-3.txt", "r", encoding="utf-8") as f:
    outputs_text = f.read()

# ============================================================
# 2. CLEAN AND PARSE INPUT FILE
# ============================================================

clean_inputs = re.sub(r"array\(", "", inputs_text)
clean_inputs = clean_inputs.replace(")", "")

inputs_by_week = ast.literal_eval(clean_inputs)

inputs_by_week = [
    [np.array(function_input, dtype=float) for function_input in week]
    for week in inputs_by_week
]

# ============================================================
# 3. CLEAN AND PARSE OUTPUT FILE
# ============================================================

clean_outputs = re.sub(r"np\.float64\((.*?)\)", r"\1", outputs_text)

outputs_by_week = ast.literal_eval(clean_outputs)

outputs_by_week = [
    np.array(week_outputs, dtype=float)
    for week_outputs in outputs_by_week
]

print(f"Loaded {len(inputs_by_week)} weeks of inputs")
print(f"Loaded {len(outputs_by_week)} weeks of outputs")

# ============================================================
# 4. HELPER FUNCTIONS
# ============================================================

def get_function_history(function_index):
    """
    Returns all historical inputs and outputs for one function.
    function_index is zero-based.
    """
    X = np.vstack([week[function_index] for week in inputs_by_week])
    y = np.array([week[function_index] for week in outputs_by_week])
    return X, y


def generate_candidates(X, y, function_index, n_local=10000, n_global=3000):
    """
    Generates candidate points for the next week.
    Uses stronger exploitation for Function 5 due to strong historical gains.
    """
    dim = X.shape[1]

    order = np.argsort(y)[::-1]
    best_x = X[order[0]]
    second_best_x = X[order[1]]

    # Function 5 has shown strong gains, so exploit it tightly
    if function_index == 4:
        local_radius = 0.012
        second_radius = 0.025
        global_count = 500

    # Functions 7 and 8 improved recently, so use moderate exploitation
    elif function_index in [6, 7]:
        local_radius = 0.035
        second_radius = 0.060
        global_count = 1500

    # Other functions remain uncertain, so allow broader exploration
    else:
        local_radius = 0.060
        second_radius = 0.090
        global_count = n_global

    local_best = np.clip(
        best_x + np.random.normal(0, local_radius, size=(n_local, dim)),
        0.000001,
        0.995000
    )

    local_second = np.clip(
        second_best_x + np.random.normal(0, second_radius, size=(n_local // 2, dim)),
        0.000001,
        0.995000
    )

    mix = np.random.uniform(0, 1, size=(1500, 1))

    interpolation = np.clip(
        mix * best_x + (1 - mix) * second_best_x + np.random.normal(0, 0.020, size=(1500, dim)),
        0.000001,
        0.995000
    )

    global_search = np.random.uniform(
        0.000001,
        0.995000,
        size=(global_count, dim)
    )

    candidates = np.vstack([
        local_best,
        local_second,
        interpolation,
        global_search,
        X
    ])

    return candidates, best_x


def train_bayesian_model(X, y):
    """
    Trains a Gaussian Process surrogate model.
    """
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    kernel = (
        ConstantKernel(1.0)
        * Matern(length_scale=1.0, nu=2.5)
        + WhiteKernel(noise_level=1e-5)
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        alpha=1e-6,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_scaled, y)

    return model, scaler


def acquisition_function(mu, sigma, candidates, best_x, function_index):
    """
    Uses Upper Confidence Bound:
    score = predicted mean + beta * uncertainty - distance penalty
    """

    distance = np.linalg.norm(candidates - best_x, axis=1)

    if function_index == 4:
        # Function 5: exploit heavily
        beta = 0.20
        distance_penalty = 0.15

    elif function_index in [6, 7]:
        # Functions 7 and 8: moderate exploitation
        beta = 0.80
        distance_penalty = 0.07

    else:
        # Other functions: more exploration
        beta = 1.80
        distance_penalty = 0.04

    score = mu + beta * sigma - distance_penalty * distance
    return score


def propose_next_input(function_index):
    """
    Generates the Week 8 input for one function.
    """
    X, y = get_function_history(function_index)

    model, scaler = train_bayesian_model(X, y)

    candidates, best_x = generate_candidates(X, y, function_index)

    candidates_scaled = scaler.transform(candidates)

    mu, sigma = model.predict(candidates_scaled, return_std=True)

    scores = acquisition_function(
        mu,
        sigma,
        candidates,
        best_x,
        function_index
    )

    best_candidate = candidates[np.argmax(scores)]

    return np.round(best_candidate, 6), np.max(y), X[np.argmax(y)]

# ============================================================
# 5. GENERATE WEEK 8 INPUTS
# ============================================================

week8_inputs = []

print("\nWEEK 8 BBO INPUT GENERATION")
print("===========================\n")

for function_index in range(8):
    candidate, best_output, best_input = propose_next_input(function_index)

    week8_inputs.append(candidate)

    print(f"Function {function_index + 1}")
    print(f"Best historical output: {best_output}")
    print(f"Best historical input:  {np.round(best_input, 6)}")
    print(f"Week 8 suggested input: {candidate}")
    print()

# ============================================================
# 6. PORTAL-READY FORMAT
# ============================================================

print("PORTAL-READY WEEK 8 INPUTS")
print("==========================\n")

for i, arr in enumerate(week8_inputs, start=1):
    formatted = "-".join(f"{x:.6f}" for x in arr)
    print(f"Function {i}: {formatted}")

FileNotFoundError: [Errno 2] No such file or directory: 'inputs-3.txt'

In [3]:
# ============================================================
# WEEK 8 BBO STRATEGY
# Week 1–7 inputs and outputs embedded directly
# Bayesian Optimisation + Adaptive Exploitation/Exploration
# ============================================================

import numpy as np
import warnings

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(42)

# ============================================================
# WEEK 1–7 INPUT DATA
# ============================================================

inputs_by_week = [
    # WEEK 1
    [
        np.array([0.211325, 0.788675]),
        np.array([0.723607, 0.276393]),
        np.array([0.166667, 0.500000, 0.833333]),
        np.array([0.125000, 0.375000, 0.625000, 0.875000]),
        np.array([0.875000, 0.625000, 0.375000, 0.125000]),
        np.array([0.150000, 0.350000, 0.550000, 0.750000, 0.950000]),
        np.array([0.900000, 0.100000, 0.700000, 0.300000, 0.500000, 0.800000]),
        np.array([0.111111, 0.222222, 0.333333, 0.444444, 0.555555, 0.666667, 0.777778, 0.888889])
    ],

    # WEEK 2
    [
        np.array([0.654299, 0.353479]),
        np.array([0.754299, 0.253479]),
        np.array([0.304299, 0.553479, 0.709691]),
        np.array([0.804299, 0.603479, 0.409691, 0.205016]),
        np.array([0.854299, 0.653479, 0.359691, 0.155016]),
        np.array([0.904299, 0.703479, 0.509691, 0.305016, 0.103275]),
        np.array([0.884299, 0.123479, 0.729691, 0.285016, 0.523275, 0.787492]),
        np.array([0.104299, 0.243479, 0.319691, 0.465016, 0.573275, 0.647492, 0.794889, 0.860861])
    ],

    # WEEK 3
    [
        np.array([0.582812, 0.421077]),
        np.array([0.712865, 0.284413]),
        np.array([0.118496, 0.481282, 0.876608]),
        np.array([1.000000, 0.683447, 0.334333, 0.000000]),
        np.array([0.882245, 0.615032, 0.380358, 0.114494]),
        np.array([0.000000, 0.226282, 0.564108, 0.905744, 1.000000]),
        np.array([0.905495, 0.091782, 0.689608, 0.305244, 0.491854, 0.804378]),
        np.array([0.113495, 0.214782, 0.338108, 0.437244, 0.549353, 0.673378, 0.771789, 0.898699])
    ],

    # WEEK 4
    [
        np.array([0.183704, 0.127166]),
        np.array([0.255353, 0.749841]),
        np.array([0.094906, 0.770362, 0.859589]),
        np.array([0.820856, 0.226975, 0.784625, 0.113609]),
        np.array([0.917860, 0.273128, 0.475575, 0.052446]),
        np.array([0.052509, 0.789636, 0.354234, 0.228337, 0.889991]),
        np.array([0.833796, 0.096195, 0.232612, 0.789158, 0.185769, 0.681185]),
        np.array([0.448564, 0.335191, 0.751694, 0.277261, 0.167614, 0.717472, 0.304117, 0.767059])
    ],

    # WEEK 5
    [
        np.array([0.582812, 0.421077]),
        np.array([0.684812, 0.281383]),
        np.array([0.171685, 0.508728, 0.819757]),
        np.array([0.734042, 0.687625, 0.466427, 0.303264]),
        np.array([0.568707, 0.909615, 0.957143, 0.011420]),
        np.array([0.744968, 0.084644, 0.801527, 0.169728, 0.990511]),
        np.array([1.000000, 0.229371, 0.690054, 0.244152, 0.521424, 1.000000]),
        np.array([0.022620, 0.000000, 0.288709, 0.762817, 0.551656, 0.961269, 0.843087, 1.000000])
    ],

    # WEEK 6
    [
        np.array([0.582812, 0.421077]),
        np.array([0.996744, 0.999561]),
        np.array([0.166667, 0.500000, 0.833333]),
        np.array([0.743137, 0.718436, 0.581810, 0.365984]),
        np.array([0.572707, 0.914615, 0.952143, 0.016420]),
        np.array([0.236209, 0.379873, 0.584786, 0.843391, 1.000000]),
        np.array([0.024461, 0.059677, 0.995330, 0.299568, 0.038576, 0.691027]),
        np.array([0.148189, 0.189483, 0.270757, 0.392507, 0.446351, 0.670593, 0.665417, 1.000000])
    ],

    # WEEK 7
    [
        np.array([0.582812, 0.421077]),
        np.array([0.744767, 0.322712]),
        np.array([0.167431, 0.503246, 0.833129]),
        np.array([0.692844, 0.522126, 0.400511, 0.337375]),
        np.array([0.543363, 0.994173, 0.995000, 0.003824]),
        np.array([0.236209, 0.379873, 0.584786, 0.843391, 1.000000]),
        np.array([0.194327, 0.008117, 0.940397, 0.331469, 0.120428, 0.708672]),
        np.array([0.208946, 0.148993, 0.202639, 0.316908, 0.431312, 0.503474, 0.634079, 0.995000])
    ]
]

# ============================================================
# WEEK 1–7 OUTPUT DATA
# ============================================================

outputs_by_week = [
    np.array([1.1327056288856873e-125, 0.5675786892822564, -0.032385408076210126,
              -17.20744048260943, 60.223125, -1.3287857969718009,
              0.34356041660314957, 9.0501517903694]),

    np.array([1.1867858443665185e-41, 0.2715258567299176, -0.1198581070659559,
              -13.082213230390916, 50.179390256321376, -1.5080002951000964,
              0.31639485635485903, 9.0219949134074]),

    np.array([7.99676981308551e-19, 0.5213723465552891, -0.04726977098841539,
              -25.67625344929702, 64.78245026474816, -1.7372122723701597,
              0.3507828450585503, 9.0587238074074]),

    np.array([-2.410121917336285e-100, 0.0244939290195959, -0.04207093453964322,
              -20.330739763644917, 61.278397876784794, -2.4624737429462145,
              0.0634803557047261, 8.275283250689899]),

    np.array([7.99676981308551e-19, 0.4258127251524913, -0.06232295859499999,
              -12.496845976106417, 942.2521944777988, -2.280900502246122,
              0.21828066675598462, 8.427686115001]),

    np.array([7.99676981308551e-19, -0.004122974640885927, -0.032385408076210126,
              -14.192389833871584, 943.6841142765069, -1.2257941580841207,
              0.4410410448155631, 9.3346877420455]),

    np.array([7.99676981308551e-19, 0.3799297936947829, -0.03240117791304375,
              -6.92709780967855, 1723.425126888365, -1.2923495282910902,
              0.919847131115406, 9.472143824862])
]

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def get_function_history(function_index):
    X = np.vstack([week[function_index] for week in inputs_by_week])
    y = np.array([week[function_index] for week in outputs_by_week])
    return X, y


def train_bayesian_model(X, y):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    kernel = ConstantKernel(1.0) * Matern(length_scale=1.0, nu=2.5) + WhiteKernel(noise_level=1e-5)

    model = GaussianProcessRegressor(
        kernel=kernel,
        alpha=1e-6,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_scaled, y)
    return model, scaler


def generate_candidates(X, y, function_index, n_local=10000, n_global=3000):
    dim = X.shape[1]

    order = np.argsort(y)[::-1]
    best_x = X[order[0]]
    second_best_x = X[order[1]]

    # Function-specific search behaviour
    if function_index == 4:
        # Function 5: strongest performer, exploit tightly
        local_radius = 0.010
        second_radius = 0.020
        global_count = 300

    elif function_index in [6, 7]:
        # Functions 7 and 8 are improving, exploit moderately
        local_radius = 0.030
        second_radius = 0.055
        global_count = 1200

    elif function_index == 0:
        # Function 1 is flat/no signal
        local_radius = 0.050
        second_radius = 0.070
        global_count = 1500

    else:
        # Other uncertain functions
        local_radius = 0.055
        second_radius = 0.085
        global_count = n_global

    local_best = np.clip(
        best_x + np.random.normal(0, local_radius, size=(n_local, dim)),
        0.000001,
        0.995000
    )

    local_second = np.clip(
        second_best_x + np.random.normal(0, second_radius, size=(n_local // 2, dim)),
        0.000001,
        0.995000
    )

    mix = np.random.uniform(0, 1, size=(1500, 1))

    interpolation = np.clip(
        mix * best_x + (1 - mix) * second_best_x + np.random.normal(0, 0.020, size=(1500, dim)),
        0.000001,
        0.995000
    )

    global_search = np.random.uniform(
        0.000001,
        0.995000,
        size=(global_count, dim)
    )

    candidates = np.vstack([
        local_best,
        local_second,
        interpolation,
        global_search,
        X
    ])

    return candidates, best_x


def acquisition_function(mu, sigma, candidates, best_x, function_index):
    distance = np.linalg.norm(candidates - best_x, axis=1)

    if function_index == 4:
        # Function 5: heavy exploitation
        beta = 0.15
        distance_penalty = 0.18

    elif function_index in [6, 7]:
        # Functions 7 and 8: moderate uncertainty search
        beta = 0.80
        distance_penalty = 0.07

    elif function_index == 0:
        # Function 1: still low-signal, allow some exploration
        beta = 1.20
        distance_penalty = 0.05

    else:
        # Other functions: balanced exploration
        beta = 1.60
        distance_penalty = 0.04

    score = mu + beta * sigma - distance_penalty * distance
    return score


def propose_week8_input(function_index):
    X, y = get_function_history(function_index)

    model, scaler = train_bayesian_model(X, y)

    candidates, best_x = generate_candidates(X, y, function_index)

    candidates_scaled = scaler.transform(candidates)

    mu, sigma = model.predict(candidates_scaled, return_std=True)

    scores = acquisition_function(mu, sigma, candidates, best_x, function_index)

    best_candidate = candidates[np.argmax(scores)]

    return np.round(best_candidate, 6), np.max(y), X[np.argmax(y)]


# ============================================================
# GENERATE WEEK 8 INPUTS
# ============================================================

week8_inputs = []

print("WEEK 8 BBO INPUT GENERATION")
print("===========================\n")

for function_index in range(8):
    candidate, best_output, best_input = propose_week8_input(function_index)

    week8_inputs.append(candidate)

    print(f"Function {function_index + 1}")
    print(f"Best historical output: {best_output}")
    print(f"Best historical input:  {np.round(best_input, 6)}")
    print(f"Week 8 suggested input: {candidate}")
    print()

# ============================================================
# PORTAL-READY FORMAT
# ============================================================

print("PORTAL-READY WEEK 8 INPUTS")
print("==========================\n")

for i, arr in enumerate(week8_inputs, start=1):
    formatted = "-".join(f"{x:.6f}" for x in arr)
    print(f"Function {i}: {formatted}")

WEEK 8 BBO INPUT GENERATION

Function 1
Best historical output: 7.99676981308551e-19
Best historical input:  [0.582812 0.421077]
Week 8 suggested input: [0.582812 0.421077]

Function 2
Best historical output: 0.5675786892822564
Best historical input:  [0.723607 0.276393]
Week 8 suggested input: [0.723607 0.276393]

Function 3
Best historical output: -0.032385408076210126
Best historical input:  [0.166667 0.5      0.833333]
Week 8 suggested input: [0.17358  0.509985 0.861627]

Function 4
Best historical output: -6.92709780967855
Best historical input:  [0.692844 0.522126 0.400511 0.337375]
Week 8 suggested input: [0.605953 0.443908 0.321881 0.510101]

Function 5
Best historical output: 1723.425126888365
Best historical input:  [0.543363 0.994173 0.995    0.003824]
Week 8 suggested input: [5.03541e-01 9.95000e-01 9.95000e-01 1.00000e-06]

Function 6
Best historical output: -1.2257941580841207
Best historical input:  [0.236209 0.379873 0.584786 0.843391 1.      ]
Week 8 suggested input: [